In [ ]:
image_folder = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/images"
mask_folder = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/masks"

In [1]:
import cv2
import os
import numpy as np
from tqdm import tqdm
import itertools
import time


def adjust_brightness_contrast(image, brightness=0, contrast=1.0):
    out = cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)
    return out


def add_gaussian_noise(image, sigma=0):
    if sigma == 0:
        return image
    noise = np.random.normal(0, sigma, image.shape)
    noisy = image.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)


def systematic_intensity_variation(
    input_directory,
    output_directory,
    brightness_values=(-40, 0, 40),
    contrast_values=(0.7, 1.0, 1.3),
    noise_sigmas=(0, 5, 15)
):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    valid_exts = (".png", ".tif", ".tiff")
    image_files = [f for f in os.listdir(input_directory) if f.lower().endswith(valid_exts)]

    start = time.time()
    total_saved = 0

    combinations = list(itertools.product(brightness_values, contrast_values, noise_sigmas))

    for image_file in tqdm(image_files, desc="Systematic intensity testing"):
        image_path = os.path.join(input_directory, image_file)
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        base_name, ext = os.path.splitext(image_file)

        for b, c, n in combinations:
            out = adjust_brightness_contrast(image, brightness=b, contrast=c)
            out = add_gaussian_noise(out, sigma=n)

            filename = f"{base_name}_b{b}_c{c}_n{n}{ext}"
            cv2.imwrite(os.path.join(output_directory, filename), out)

            total_saved += 1

    elapsed = time.time() - start

    print(f"Created {total_saved} images in {round(elapsed, 2)} seconds.")

    return {
        "input_images": len(image_files),
        "total_generated": total_saved,
        "combinations_per_image": len(combinations),
        "time_spent_s": round(elapsed, 2),
    }


In [2]:
brightness_values = (-40, 0, 40)
contrast_values   = (0.7, 1.0, 1.3)
noise_sigmas      = (0, 5, 15)

In [3]:
input_directory = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/images"
output_directory = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/aug"

systematic_intensity_variation(input_directory, output_directory)

Systematic intensity testing:   0%|          | 0/4 [00:00<?, ?it/s]


RuntimeError: Unable to configure default ndarray.__repr__